In [1]:
import torch
import pandas as pd 

dataset=pd.read_csv('Telco-Customer-Churn.csv')
# dataset.head()

In [2]:
dataset=dataset.drop(['customerID'],axis=1)
dataset.columns=dataset.columns.str.lower()
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   seniorcitizen     7043 non-null   int64  
 2   partner           7043 non-null   object 
 3   dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   phoneservice      7043 non-null   object 
 6   multiplelines     7043 non-null   object 
 7   internetservice   7043 non-null   object 
 8   onlinesecurity    7043 non-null   object 
 9   onlinebackup      7043 non-null   object 
 10  deviceprotection  7043 non-null   object 
 11  techsupport       7043 non-null   object 
 12  streamingtv       7043 non-null   object 
 13  streamingmovies   7043 non-null   object 
 14  contract          7043 non-null   object 
 15  paperlessbilling  7043 non-null   object 
 16  paymentmethod     7043 non-null   object 


In [65]:
dataset.head()

,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
dataset['gender']=dataset['gender'].map({ 'Male':1,'Female':0 })

In [4]:
from sklearn.preprocessing import LabelEncoder
labelencoder=LabelEncoder()
dataset['partner']=labelencoder.fit_transform(dataset['partner'])

In [5]:
columns=['dependents','phoneservice','multiplelines','paperlessbilling','churn']
for column in columns:
    dataset[column]=labelencoder.fit_transform(dataset[column])

In [6]:
from sklearn.preprocessing import OneHotEncoder 
encoder=OneHotEncoder(sparse_output=False)
internet=encoder.fit_transform(dataset[['internetservice']])
dataset=dataset.drop(['internetservice'],axis=1)
encoder.get_feature_names_out()
internetservice=pd.DataFrame(internet,columns=encoder.get_feature_names_out(['internetservice']))

In [7]:
dataset=pd.concat((dataset,internetservice),axis=1)
# dataset.head()

In [8]:
column=['onlinesecurity','onlinebackup','deviceprotection','techsupport','streamingtv','streamingmovies','contract','paymentmethod']
for col in column:
    colum=encoder.fit_transform(dataset[[col]])
    dataset=dataset.drop([col],axis=1)
    dataframe=pd.DataFrame(colum,columns=encoder.get_feature_names_out([col]))
    dataset=pd.concat((dataset,dataframe),axis=1)
    

In [72]:
for column in dataset.columns:
    print(column,dataset[column].nunique())

gender 2
seniorcitizen 2
partner 2
dependents 2
tenure 73
phoneservice 2
multiplelines 3
paperlessbilling 2
monthlycharges 1585
totalcharges 6531
churn 2
internetservice_DSL 2
internetservice_Fiber optic 2
internetservice_No 2
onlinesecurity_No 2
onlinesecurity_No internet service 2
onlinesecurity_Yes 2
onlinebackup_No 2
onlinebackup_No internet service 2
onlinebackup_Yes 2
deviceprotection_No 2
deviceprotection_No internet service 2
deviceprotection_Yes 2
techsupport_No 2
techsupport_No internet service 2
techsupport_Yes 2
streamingtv_No 2
streamingtv_No internet service 2
streamingtv_Yes 2
streamingmovies_No 2
streamingmovies_No internet service 2
streamingmovies_Yes 2
contract_Month-to-month 2
contract_One year 2
contract_Two year 2
paymentmethod_Bank transfer (automatic) 2
paymentmethod_Credit card (automatic) 2
paymentmethod_Electronic check 2
paymentmethod_Mailed check 2


In [73]:
dataset.head()

,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,paperlessbilling,monthlycharges,totalcharges,...,streamingmovies_No,streamingmovies_No internet service,streamingmovies_Yes,contract_Month-to-month,contract_One year,contract_Two year,paymentmethod_Bank transfer (automatic),paymentmethod_Credit card (automatic),paymentmethod_Electronic check,paymentmethod_Mailed check
0,0,0,1,0,1,0,1,1,29.85,29.85,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,0,0,0,34,1,0,0,56.95,1889.5,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,1,0,0,0,2,1,0,1,53.85,108.15,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1,0,0,0,45,0,1,0,42.30,1840.75,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,0,0,0,0,2,1,0,1,70.70,151.65,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [9]:
import numpy as np
charges=['monthlycharges','totalcharges']
for charge in charges:
    dataset[charge]=pd.to_numeric(dataset[charge],errors='coerce')
    dataset[charge]=dataset[charge].replace(0,np.nan)
    dataset[charge]=dataset[charge].fillna(dataset[charge].median())
    # print(np.isnan(dataset[charge]).sum())   


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
ind_variable=dataset.drop('churn',axis=1)
dep_variable=dataset['churn']
scaler=StandardScaler()
x_train,x_test,y_train,y_test=train_test_split(ind_variable,
                                               dep_variable,
                                               test_size=0.2,
                                               random_state=42,
                                               stratify=dep_variable)
x_train=scaler.fit_transform(x_train)
x_test=scaler.fit_transform(x_test)
# print(np.isnan(x_train).sum())

In [15]:
import torch 
import torch.nn as nn 
import torch.optim as optim
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.input=nn.Linear(38,64)
        self.relu=nn.ReLU()
        self.h1=nn.Linear(64,32)
        self.h2=nn.Linear(32,16)
        # self.h3=nn.Linear(32,16)
        self.output=nn.Linear(16,1)
        self.sigmoid=nn.Sigmoid()

    def forward(self,x):
        x=self.input(x)
        x=self.relu(x)
        x=self.h1(x)
        x=self.relu(x)
        x=self.h2(x)
        x=self.relu(x)
        # x=self.h3(x)
        # x=self.relu(x)
        x=self.output(x)
        x=self.sigmoid(x)
     
        return x

print('success')

success


In [ ]:
y_train=np.array(y_train)
y_test=np.array(y_test)

y_train=torch.tensor(y_train,dtype=torch.float32).reshape(-1,1)
y_test=torch.tensor(y_test,dtype=torch.float32).reshape(-1,1)
x_test=torch.tensor(x_test,dtype=torch.float32)
x_train=torch.tensor(x_train,dtype=torch.float32)

In [16]:


x_training,x_val,y_training,y_val=train_test_split(x_train,
                                                   y_train,
                                                   test_size=0.2,
                                                   random_state=42,
                                                   stratify=y_train)
                                                
                                             

model=Model()

criteria=nn.BCELoss()

optimizer=optim.Adam(
    model.parameters(),
    lr=0.001)

epochs=100

for epoch in range(epochs):
    predictions=model(x_training)
    loss=criteria(predictions,y_training)
    pred_prob=(predictions>0.4).float()
    accuracy=(pred_prob == y_training).float().mean()
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()
    with torch.no_grad():
        validations=model(x_val)
        val_loss=criteria(validations,y_val)
        val_prob=(validations>0.4).float()
        val_accuracy=(val_prob==y_val).float().mean()
    print(f'Epoch:{epoch}   '
         f'Loss:{loss.item():.4f}  '
         f'Accuracy:{accuracy.item():.4f}  '
         f'Val_loss:{val_loss.item():.4f}  '
         f'Val_accuracy:{val_accuracy.item():.4f} ')

with torch.no_grad():
    final_prediction=model(x_test)
    final_loss=criteria(final_prediction,y_test)
    fin_prob=(final_prediction>0.4).float()
    final_accuracy=(fin_prob == y_test).float().mean()

    print(f'Test Loss: {final_loss.item():.4f}\n'
          f'Test Accuracy:{final_accuracy.item():.4f}')
          
    


Epoch:0   Loss:0.7490  Accuracy:0.2654  Val_loss:0.7428  Val_accuracy:0.2653 
Epoch:1   Loss:0.7431  Accuracy:0.2654  Val_loss:0.7369  Val_accuracy:0.2653 
Epoch:2   Loss:0.7372  Accuracy:0.2654  Val_loss:0.7308  Val_accuracy:0.2653 
Epoch:3   Loss:0.7312  Accuracy:0.2654  Val_loss:0.7246  Val_accuracy:0.2653 
Epoch:4   Loss:0.7250  Accuracy:0.2654  Val_loss:0.7182  Val_accuracy:0.2653 
Epoch:5   Loss:0.7186  Accuracy:0.2654  Val_loss:0.7118  Val_accuracy:0.2653 
Epoch:6   Loss:0.7123  Accuracy:0.2654  Val_loss:0.7053  Val_accuracy:0.2653 
Epoch:7   Loss:0.7059  Accuracy:0.2654  Val_loss:0.6989  Val_accuracy:0.2653 
Epoch:8   Loss:0.6996  Accuracy:0.2654  Val_loss:0.6924  Val_accuracy:0.2653 
Epoch:9   Loss:0.6932  Accuracy:0.2654  Val_loss:0.6857  Val_accuracy:0.2653 
Epoch:10   Loss:0.6865  Accuracy:0.2654  Val_loss:0.6788  Val_accuracy:0.2653 
Epoch:11   Loss:0.6796  Accuracy:0.2654  Val_loss:0.6715  Val_accuracy:0.2653 
Epoch:12   Loss:0.6723  Accuracy:0.2654  Val_loss:0.6638  Val_

In [17]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test,fin_prob))

[[854 181]
 [133 241]]


In [18]:
from sklearn.metrics import classification_report
print(classification_report(y_test,fin_prob))

              precision    recall  f1-score   support

         0.0       0.87      0.83      0.84      1035
         1.0       0.57      0.64      0.61       374

    accuracy                           0.78      1409
   macro avg       0.72      0.73      0.73      1409
weighted avg       0.79      0.78      0.78      1409

